# 4단계: 속성별 감성분석 (ABSA) — 조 담당 (생활, 화장품)

3단계에서 이미 확보한 (리뷰, 속성) 정답 쌍에 감성(긍정/부정/중립)을 예측하는 분류기를 학습한다. AIHub 원본에 감성 정답이 이미 있으니, 3단계 모델의 예측이 아니라 `aspects.parquet`의 `SentimentPolarity`를 그대로 학습 라벨로 쓴다.

**입력 방식**: 문장쌍(sentence-pair) — `(속성명, 리뷰전문)`을 같이 넣어서 "이 리뷰에서 이 속성에 대한 감성"을 예측하게 한다. 3단계(리뷰 하나로 여러 속성 존재여부 동시예측)와 다르게, 여기서는 리뷰 하나가 속성 개수만큼 여러 학습 샘플로 쪼개진다.

**클래스 불균형은 시작 전에 이미 확인함**: 생활/화장품 둘 다 긍정 약 70%, 부정 23~25%, 중립 6%로 꽤 치우쳐 있다(로컬 EDA로 사전 확인, `data/processed/absa_datasets/{도메인}_absa.parquet` 생성 시 class_weight도 같이 계산해둠). 3단계 화장품에서 겪은 시행착오(가중치 0→과교정→적정선)를 반복 안 하려고, 이번엔 처음부터 balanced 공식(`n/(3*count)`)으로 가중치를 계산해서 적용한다.

## Colab(GPU)에서 실행

**실행 전 체크리스트**
1. 런타임 > 런타임 유형 변경 → GPU
2. 왼쪽 파일 탭에 아래 파일 업로드
   - `data/processed/absa_datasets/{도메인}_absa.parquet`
3. 아래 `DOMAIN` 변수를 `생활` 또는 `화장품`으로 설정
4. 끝까지 실행 → 마지막 셀에서 결과 자동 다운로드

In [ ]:
!pip install -q "transformers>=4.40" "accelerate>=1.1.0" scikit-learn pyarrow

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from sklearn.metrics import f1_score, classification_report

# ===== 본인이 학습할 도메인으로 변경 =====
DOMAIN = "생활"   # "생활" | "화장품"
MODEL_NAME = "klue/bert-base"
DATA_DIR = "."      # 업로드한 파일이 있는 경로
OUT_DIR = f"./{DOMAIN}_absa_model_output"
POLARITY_NAMES = ["부정", "중립", "긍정"]  # label 0,1,2

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "GPU가 안 잡혔다. 런타임 > 런타임 유형 변경에서 GPU를 선택했는지 확인할 것"

## 1. 데이터 로드

`data/processed/absa_datasets/{도메인}_absa.parquet`는 로컬에서 미리 만들어뒀다: `aspects.parquet`(정답 SentimentPolarity)에서 도메인 필터링 → `reviews.parquet`(정제 텍스트) 조인 → `aspect_datasets/{도메인}_reviews.parquet`의 재분할 `split` 컬럼 조인(3단계와 동일 분할 유지, 새로 나누지 않음). `label`은 SentimentPolarity(-1/0/1)를 (0/1/2)로 재인코딩한 것.

**주의**: `aspects.parquet` 자체에도 구(원본) split 컬럼이 있어서, 조인 전에 버리고 재분할 split으로 교체해야 한다 — 안 그러면 조용히 옛날 분할을 쓰게 됨(로컬에서 실제로 이 버그를 만났다가 고침).

In [ ]:
df = pd.read_parquet(f"{DATA_DIR}/{DOMAIN}_absa.parquet")

train_df = df[df["split"] == "Training"].reset_index(drop=True)
val_df = df[df["split"] == "Validation"].reset_index(drop=True)

print(f"도메인={DOMAIN}")
print(f"Train={len(train_df)}, Val={len(val_df)}")

print("\nTrain 라벨 분포:")
print(train_df["label"].value_counts().sort_index().rename(index=lambda i: POLARITY_NAMES[i]))
print("\nVal 라벨 분포:")
print(val_df["label"].value_counts().sort_index().rename(index=lambda i: POLARITY_NAMES[i]))

## 2. 모델/데이터셋 준비

`klue/bert-base` + 문장쌍(`(속성명, 리뷰텍스트)`) 입력, 3-class(부정/중립/긍정) 단일라벨 분류. 3단계는 multi-label(sigmoid+BCE)이었는데 이번엔 클래스가 서로 배타적(한 속성-리뷰 쌍의 감성은 하나)이라 표준 softmax+CrossEntropyLoss 구조를 쓴다.

**클래스 불균형 대응**: 3단계는 "가중치 없이 학습 → 실패 확인 → 상한 20 → 과교정 확인 → 상한 5"로 세 번 돌려서 찾았는데, 이번엔 처음부터 불균형을 알고 시작하니 sklearn 스타일 balanced 공식(`n_train/(3*count)`)으로 한 번에 적용한다. 과교정 여부는 결과 나오면 라벨별 precision/recall로 확인.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ABSADataset(Dataset):
    def __init__(self, frame):
        self.aspects = frame["Aspect"].tolist()
        self.texts = frame["RawText_clean"].tolist()
        self.labels = frame["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # 문장쌍: (속성명, 리뷰전문) -> token_type_ids로 구분됨
        enc = tokenizer(self.aspects[idx], self.texts[idx], truncation=True, max_length=128, padding="max_length")
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_ds = ABSADataset(train_df)
val_ds = ABSADataset(val_df)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
)

# ===== 클래스 불균형 대응: balanced class weight =====
n_train = len(train_df)
counts = train_df["label"].value_counts().sort_index()
class_weight = n_train / (3 * counts)
class_weight_tensor = torch.tensor(class_weight.values, dtype=torch.float32)
print("class_weight:", {POLARITY_NAMES[i]: round(w, 3) for i, w in zip(counts.index, class_weight.values)})


class WeightedTrainer(Trainer):
    """기본 Trainer는 클래스 가중치 없는 CrossEntropyLoss를 씀 -> class_weight 적용한 버전으로 교체."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        target = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weight_tensor.to(logits.device))
        loss = loss_fct(logits, target)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=1)
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

## 3. 학습

3단계보다 데이터가 더 많다(리뷰 단위가 아니라 속성-감성 쌍 단위라 생활 14.7만/화장품 16.5만 건). epoch=3, batch=32, lr=2e-5는 3단계와 동일한 시작값.

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to=[],
    fp16=True,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## 4. 결과 검증

3-class(부정/중립/긍정)라 3단계보다 클래스 수는 적지만, 중립이 원래 6%밖에 안 되는 소수 클래스라 이 클래스가 제대로 학습됐는지가 핵심 체크포인트다.

In [ ]:
pred_output = trainer.predict(val_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=1)

print("=== 전체 지표 ===")
print(pred_output.metrics)

print("\n=== 클래스별 지표 ===")
print(classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=POLARITY_NAMES, zero_division=0))

never_predicted = [POLARITY_NAMES[i] for i in range(3) if (y_pred == i).sum() == 0]
print(f"\n검증셋에서 단 한 번도 예측 안 된 클래스: {never_predicted if never_predicted else '없음'}")

print("\n=== 샘플 예측 육안 확인 (5건) ===")
rng = np.random.default_rng(0)
for i in rng.choice(len(val_df), size=5, replace=False):
    row = val_df.iloc[i]
    print(f"\n속성: {row['Aspect']} | 리뷰: {row['RawText_clean'][:60]}")
    print(f"  실제: {POLARITY_NAMES[y_true[i]]} / 예측: {POLARITY_NAMES[y_pred[i]]}")

## 5. 저장

3단계와 동일하게, 모델 가중치는 구글드라이브에(필요시), 가벼운 결과물(라벨/지표)은 로컬로 다운로드.

In [ ]:
import os

trainer.save_model(f"{OUT_DIR}/final")
tokenizer.save_pretrained(f"{OUT_DIR}/final")

report_dict = classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=POLARITY_NAMES, zero_division=0, output_dict=True)
with open(f"{DOMAIN}_absa_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "domain": DOMAIN,
        "overall": pred_output.metrics,
        "per_class": report_dict,
        "never_predicted_classes": never_predicted,
        "class_weight_used": {POLARITY_NAMES[i]: float(w) for i, w in zip(counts.index, class_weight.values)},
    }, f, ensure_ascii=False, indent=2)

print("저장 완료:", f"{DOMAIN}_absa_metrics.json")

from google.colab import files
files.download(f"{DOMAIN}_absa_metrics.json")

print("\n모델 가중치 위치:", f"{OUT_DIR}/final")